In [ ]:
import sys
import os

os.chdir("../..")
sys.path.append("./VeloEdit")

import json
import pandas as pd
import torch
from PIL import Image
from diffusers.utils import make_image_grid
from IPython.display import display, clear_output

# Shared relative paths
DATASET_DIR = "./datasets/test_images"
OUTPUT_DIR  = "./outputs/veloedit/1d"
CSV_PATH    = "./SliderEdit/experiments/experiments24-7.csv"

print("Working Directory:", os.getcwd())
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

### Load FLUX

In [ ]:
from transformers import CLIPTextModel, T5EncoderModel, BitsAndBytesConfig
from diffusers import AutoencoderKL, FluxTransformer2DModel, FluxPipeline

# Configure 4-bit quantization matching your setup
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

MODEL_ID = "black-forest-labs/FLUX.1-Kontext-dev"

clip_text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.bfloat16).to("cuda")
t5_text_encoder = T5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder_2", torch_dtype=torch.bfloat16).to("cuda")
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to("cuda", dtype=torch.bfloat16)

transformer = FluxTransformer2DModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    quantization_config=quantization_config,
    device_map={"": "cuda"}
)

pipe = FluxPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    text_encoder=clip_text_encoder,
    text_encoder_2=t5_text_encoder,
    transformer=transformer,
    torch_dtype=torch.bfloat16
)

print("FLUX Pipeline successfully initialized for VeloEdit!")

### Load CSV

In [ ]:
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Cannot find CSV at {CSV_PATH}.")

df_experiments = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df_experiments)} experiments from {CSV_PATH}\n")
display(df_experiments[["id", "domain", "subprompt_1", "subprompt_2", "test_focus"]].head(10))

### Benchmark VeloEdit

In [ ]:
def run_veloedit_suite(
    df,
    pipe,
    intensity_steps=[0.0, 0.25, 0.50, 0.75, 1.00],
    output_root="./outputs/veloedit/1d",
    source_img_dir="./datasets/test_images"
):
    output_root_abs = os.path.abspath(output_root)
    source_img_dir_abs = os.path.abspath(source_img_dir)
    os.makedirs(output_root_abs, exist_ok=True)

    print("==================================================")
    print("  STARTING VELOEDIT BENCHMARK RUN (1D)")
    print(f"  Output Directory: {output_root_abs}")
    print(f"  Intensity Steps: {intensity_steps}")
    print("==================================================\n")

    for idx, row in df.iterrows():
        sweep_id = str(row['id'])
        subprompt1 = str(row['subprompt_1'])
        seed = int(row['seed'])
        base_prompt = str(row['base_prompt'])
        test_focus = str(row['test_focus'])

        exp_dir = os.path.join(output_root_abs, sweep_id)
        os.makedirs(exp_dir, exist_ok=True)

        # Step 1: Read Pre-generated Base Source Image from SliderEdit
        source_img_path = os.path.join(source_img_dir_abs, f"{sweep_id}.png")
        if not os.path.exists(source_img_path):
            print(f"[WARNING] Baseline image missing for '{sweep_id}'. Ensure SliderEdit finishes running first!")
            continue
        
        img = Image.open(source_img_path).convert("RGB")

        # Save metadata
        meta_data = {
            "id": sweep_id,
            "domain": row['domain'],
            "paradigm": "veloedit",
            "mode": "1d",
            "base_prompt": base_prompt,
            "subprompt_1": subprompt1,
            "seed": seed,
            "intensity_steps": intensity_steps,
            "test_focus": test_focus
        }
        with open(os.path.join(exp_dir, "meta.json"), "w") as f:
            json.dump(meta_data, f, indent=4)

        # Step 2: Run Velocity Edit Sweeps
        outputs = []
        clean_prompt = "".join(c for c in subprompt1 if c.isalnum() or c in (" ", "_")).replace(" ", "_")
        print(f"\n[{idx+1}/{len(df)}] Running VeloEdit for '{sweep_id}': '{subprompt1}'")

        for val in intensity_steps:
            generator = torch.Generator(device="cuda").manual_seed(seed)
            
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                # 💡 Calling VeloEdit velocity intervention hook/wrapper
                out_img = pipe(
                    image=img,
                    prompt=subprompt1,
                    generator=generator,
                    # edit_scale=val  <-- Check VeloEdit API for exact scale param name
                ).images[0]

            val_str = f"{val:.2f}".replace("-", "neg_")
            fname = f"{sweep_id}_{clean_prompt}_intensity_{val_str}.png"
            full_save_path = os.path.join(exp_dir, fname)
            
            out_img.save(full_save_path)
            outputs.append(out_img)
            print(f" -> Saved: {full_save_path}")

        grid = make_image_grid([x.resize((128, 128)) for x in outputs], rows=1, cols=len(intensity_steps))
        display(grid)

        del outputs
        del grid
        torch.cuda.empty_cache()

    print("\nVeloEdit Benchmark Run Complete!")

In [ ]:
VELOEDIT_STEPS = [0.0, 0.25, 0.50, 0.75, 1.00]

run_veloedit_suite(
    df=df_experiments,
    pipe=pipe,
    intensity_steps=VELOEDIT_STEPS
)